# Preprocesado de los datos
### Proyecto de Aprendizaje Automático · Encuesta de salud del consorcio de atención primaria

Este notebook ejecuta, paso a paso, el preprocesado definido a partir del EDA (`01_eda.ipynb`, sección 10). Toda la lógica vive en `src/preprocesamiento.py`, de modo que **el mismo código** se reutilizará en el entrenamiento, en la evaluación y en la API.

**Principio de diseño: separar lo determinista de lo aprendido**

| Tipo | Qué hace | Dónde | ¿Aprende de los datos? |
|---|---|---|---|
| Limpieza | Elimina PII y gestión, unifica binarias, codifica ordinales | `pp.limpiar` | No |
| Variables derivadas | `imc_categoria`, `n_factores_riesgo_cv`, indicadores de nulos | `pp.anadir_features` | No |
| Imputación y escalado | Mediana/moda + estandarización | `pp.construir_preprocesador` | **Sí → solo con train** |

Las dos primeras se pueden aplicar a cualquier dato nuevo. La tercera se ajusta **exclusivamente con el conjunto de entrenamiento** para no filtrar información del test, y se guarda como artefacto (`joblib`) que recibe datos **crudos** y devuelve la matriz lista para el modelo.

**Salidas**
- `data/processed/diabetes_limpio.parquet` (+ `.csv`): dataset limpio, sin imputar, para análisis y perfilado de clústeres.
- `data/processed/{clasificacion,regresion}/`: `X_train`, `X_test`, `y_train`, `y_test` ya transformados.
- `data/processed/clustering/X_clustering.parquet`: matriz estandarizada para clusterizar.
- `models/preprocesadores/{clasificacion,regresion,clustering}.joblib`: pipelines ajustados.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src import preprocesamiento as pp

pd.set_option("display.max_columns", 50)
pp.RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)
pp.RUTA_PREPROCESADORES.mkdir(parents=True, exist_ok=True)

## 1. Carga de los datos crudos

In [2]:
crudo = pp.cargar_datos()
print(f"{crudo.shape[0]:,} filas × {crudo.shape[1]} columnas")
crudo.head(3)

12,750 filas × 29 columnas


,id_paciente_proy,dni,nombre,apellidos,direccion,telefono,email,num_historia_clinica,presion_alta,colesterol_alto,chequeo_colesterol_5a,imc,fumador,ictus,enfermedad_cardiaca,actividad_fisica,consume_fruta,consume_verdura,consumo_alcohol_alto,salud_general,dias_salud_mental_mala,dias_salud_fisica_mala,dificultad_caminar,sexo,grupo_edad,nivel_educativo,nivel_ingresos,diabetes,usuario_alta_datos
0,90,9002b707403a13310b63559f1424a2a3,55f15684574c8895c4234fb7987d23cb,cde3522f31388e7cfe8663f173e07bc0,5a0d0afae7c4f2b727332e30663222ee,16a767cbb4bbeb01d995a44e320ac087,9ba86cc234d2bc2067fec10ff049a7aa,da2d03b68a2745843ee1060fc98a346c,si,si,si,NaN,no,no,no,si,si,si,no,NaN,NaN,0,no,hombre,55-59,secundaria,NaN,no,admin
1,2443,162319c0551225e9fe2c5da7b6d52e71,30c9a0e33312b54775ee3fc1d34483d7,86996ec928b462ee8f0343ffbe0f82b1,c16b76aaa2d4abca01b67816003c67a2,7176ee8d6dae058edcb75528c0d74951,84077d4793dc2403eb934b9ad8490187,97b18957d6589adf7a7fd119a8959ef8,si,si,si,41.0,NO,no,no,NO,SI,si,no,buena,0.0,0,SI,mujer,80+,secundaria,10-15k,si,us_admision_1
2,11061,e7fee47d66a627a2bd11e1abcb6fc71d,fa9b67a5f04ab381a655ce11f71ede16,1de54ad005a1c2e875664eb4e3e3b0e3,159882b4d4946b7ae0cb163e731af530,ffb6bb2a2189224fd204a4e7015ddac1,674f85328d69d9c64b29aa1f05fe42d1,2e2922cdd75430e622f1540f5614679f,si,si,si,30.0,no,no,no,si,no,si,no,muy buena,0.0,0,no,hombre,60-64,secundaria,35-50k,si,us_admision_1


## 2. Limpieza determinista (`pp.limpiar`)
1. `id_paciente_proy` pasa a ser el índice (trazabilidad sin que el modelo lo vea).
2. Se eliminan las 7 columnas PII y `usuario_alta_datos`.
3. Las 12 binarias (incluido el target `diabetes`) se unifican a 0/1 (`si/SI/True/1.0 → 1`, `no/NO/False/0.0 → 0`).
4. `sexo → es_hombre` (0/1).
5. Ordinales a enteros: `grupo_edad` 1–13, `salud_general` 1 (excelente)–5 (mala), `nivel_educativo` 1–6, `nivel_ingresos` 1–8.
6. Numéricas fuera de rango plausible (IMC ∉ [10, 100], días ∉ [0, 30]) → nulo (no hay ninguna en este archivo).

In [3]:
limpio = pp.limpiar(crudo)
print(f"Columnas: {crudo.shape[1]} → {limpio.shape[1]} (+ índice)")
print("Eliminadas:", sorted(set(crudo.columns) - set(limpio.columns) - {pp.COL_ID, "sexo"}))

# Antes / después para un registro con codificaciones no estándar
ejemplo = crudo.loc[crudo["fumador"].isin(["True", "1.0", "SI"])].iloc[0]
fila_cruda = ejemplo.drop(pp.COLS_PII + pp.COLS_GESTION + [pp.COL_ID]).rename({"sexo": "es_hombre"})
comparacion = pd.DataFrame({"crudo": fila_cruda, "limpio": limpio.loc[ejemplo[pp.COL_ID]]}).loc[limpio.columns]
comparacion.index.name = f"paciente {ejemplo[pp.COL_ID]}"
comparacion

Columnas: 29 → 20 (+ índice)
Eliminadas: ['apellidos', 'direccion', 'dni', 'email', 'nombre', 'num_historia_clinica', 'telefono', 'usuario_alta_datos']


,crudo,limpio
paciente 8382,,
imc,23.0,23.0
dias_salud_mental_mala,2.0,2.0
grupo_edad,55-59,8.0
salud_general,muy buena,2.0
nivel_educativo,titulado superior,6.0
nivel_ingresos,>75k,8.0
presion_alta,si,1.0
colesterol_alto,si,1.0
chequeo_colesterol_5a,si,1.0


In [4]:
# Comprobaciones de la limpieza
binarias = pp.COLS_BINARIAS + [pp.TARGET_CLASIF]
assert not set(pp.COLS_PII + pp.COLS_GESTION) & set(limpio.columns), "Quedan columnas PII o de gestión"
assert limpio[binarias].isin([0, 1, np.nan]).all().all(), "Hay binarias con valores distintos de 0/1/NaN"
assert limpio.index.is_unique
assert (crudo.isna().sum().sum() == limpio.isna().sum().sum()), "La limpieza ha creado o perdido nulos"
for col, mapa in pp.MAPAS_ORDINALES.items():
    assert limpio[col].dropna().isin(mapa.values()).all()
print("✔ Limpieza correcta: sin PII, binarias en {0, 1}, ordinales en su rango y mismos nulos que el original.")
limpio.describe().T[["count", "mean", "min", "max"]]

✔ Limpieza correcta: sin PII, binarias en {0, 1}, ordinales en su rango y mismos nulos que el original.


,count,mean,min,max
imc,12353.0,29.124099,13.0,95.0
dias_salud_mental_mala,12353.0,3.418198,0.0,30.0
grupo_edad,12750.0,8.372314,1.0,13.0
salud_general,12483.0,2.688777,1.0,5.0
nivel_educativo,12750.0,4.970980,1.0,6.0
nivel_ingresos,12093.0,5.859340,1.0,8.0
presion_alta,12750.0,0.501882,0.0,1.0
colesterol_alto,12750.0,0.482275,0.0,1.0
chequeo_colesterol_5a,12750.0,0.968941,0.0,1.0
fumador,12374.0,0.464199,0.0,1.0


## 3. Variables derivadas (`pp.anadir_features`)
- `imc_categoria` (1–6): categoría OMS del IMC, más interpretable para describir clústeres.
- `n_factores_riesgo_cv` (0–5): nº de factores de riesgo cardiovascular (hipertensión, colesterol, tabaco, cardiopatía, ictus). En el EDA es el 2º predictor más asociado a diabetes (V = 0,38).
- `falta_habitos`, `falta_imc`: indicadores de los dos patrones de nulos.

Ninguna usa los objetivos, así que no introducen fuga.

In [5]:
datos = pp.anadir_features(limpio)
datos[pp.COLS_FEATURES_NUEVAS_ORD + pp.COLS_FEATURES_NUEVAS_BIN].describe().T[["count", "mean", "min", "max"]]

,count,mean,min,max
imc_categoria,12353.0,3.373027,1.0,6.0
n_factores_riesgo_cv,12750.0,1.613569,0.0,5.0
falta_habitos,12750.0,0.029490,0.0,1.0
falta_imc,12750.0,0.031137,0.0,1.0


In [6]:
datos.to_parquet(pp.RUTA_PROCESADOS / "diabetes_limpio.parquet")
datos.to_csv(pp.RUTA_PROCESADOS / "diabetes_limpio.csv")
print(f"Guardado diabetes_limpio: {datos.shape[0]:,} filas × {datos.shape[1]} columnas (nulos sin imputar: {datos.isna().sum().sum():,})")

Guardado diabetes_limpio: 12,750 filas × 24 columnas (nulos sin imputar: 3,619)


## 4. Particiones y preprocesadores para las líneas supervisadas

- **Partición 80/20** con semilla fija (`SEMILLA = 42`). El 20 % de test se reserva para la evaluación final de cada línea; la selección de modelos se hará con validación cruzada dentro de train.
- **Clasificación**: estratificada por `diabetes` (mantiene el 33 % en ambos conjuntos).
- **Regresión**: estratificada por tramos del objetivo (**0 / 1–29 / 30**), para conservar la forma cero-inflada y bimodal.
- Los preprocesadores (`Pipeline`: limpieza → imputación → escalado) se **ajustan solo con train** y se aplican a ambos conjuntos.

| Bloque | Columnas | Transformación |
|---|---|---|
| Continuas | `imc`, `dias_salud_mental_mala` (+ `dias_salud_fisica_mala` en clasificación) | Mediana + `StandardScaler` |
| Ordinales | edad, salud general, educación, ingresos, `imc_categoria`, `n_factores_riesgo_cv` | Mediana + `StandardScaler` |
| Binarias | 12 binarias + indicadores de nulos (+ `diabetes` en regresión) | Moda (se quedan en 0/1) |

In [7]:
resumen = []
for linea in ("clasificacion", "regresion"):
    X_train_crudo, X_test_crudo, y_train, y_test = pp.dividir(crudo, linea)

    preprocesador = pp.construir_preprocesador(linea).fit(X_train_crudo)  # ajuste SOLO con train
    X_train = preprocesador.transform(X_train_crudo)
    X_test = preprocesador.transform(X_test_crudo)

    carpeta = pp.RUTA_PROCESADOS / linea
    carpeta.mkdir(parents=True, exist_ok=True)
    X_train.to_parquet(carpeta / "X_train.parquet")
    X_test.to_parquet(carpeta / "X_test.parquet")
    y_train.to_frame().to_parquet(carpeta / "y_train.parquet")
    y_test.to_frame().to_parquet(carpeta / "y_test.parquet")
    joblib.dump(preprocesador, pp.RUTA_PREPROCESADORES / f"{linea}.joblib")

    resumen.append({"línea": linea, "X_train": X_train.shape, "X_test": X_test.shape,
                    "media y_train": y_train.mean(), "media y_test": y_test.mean(),
                    "nulos tras transformar": int(X_train.isna().sum().sum() + X_test.isna().sum().sum())})
pd.DataFrame(resumen).set_index("línea")

,X_train,X_test,media y_train,media y_test,nulos tras transformar
línea,,,,,
clasificacion,"(10200, 23)","(2550, 23)",0.332353,0.332549,0
regresion,"(10200, 23)","(2550, 23)",4.975784,4.995686,0


### 4.1 Comprobación de las particiones

In [8]:
y_tr_c = pd.read_parquet(pp.RUTA_PROCESADOS / "clasificacion" / "y_train.parquet")["diabetes"]
y_te_c = pd.read_parquet(pp.RUTA_PROCESADOS / "clasificacion" / "y_test.parquet")["diabetes"]
y_tr_r = pd.read_parquet(pp.RUTA_PROCESADOS / "regresion" / "y_train.parquet")["dias_salud_fisica_mala"]
y_te_r = pd.read_parquet(pp.RUTA_PROCESADOS / "regresion" / "y_test.parquet")["dias_salud_fisica_mala"]

balance = pd.DataFrame({
    "train": [y_tr_c.mean(), *pp.tramos_regresion(y_tr_r).value_counts(normalize=True).sort_index()],
    "test": [y_te_c.mean(), *pp.tramos_regresion(y_te_r).value_counts(normalize=True).sort_index()],
}, index=["% diabetes", "% días = 0", "% días 1–29", "% días = 30"])
display(balance.style.format("{:.2%}"))

assert abs(y_tr_c.mean() - y_te_c.mean()) < 0.01
assert y_tr_r.between(0, 30).all() and y_te_r.between(0, 30).all()
assert not set(y_tr_c.index) & set(y_te_c.index), "Hay pacientes en train y test a la vez"
print("✔ Particiones estratificadas, disjuntas y con el objetivo de regresión en [0, 30].")

,train,test
% diabetes,33.24%,33.25%
% días = 0,60.33%,60.31%
% días 1–29,30.32%,30.35%
% días = 30,9.34%,9.33%


✔ Particiones estratificadas, disjuntas y con el objetivo de regresión en [0, 30].


### 4.2 Qué ha aprendido el preprocesador (solo de train)

In [9]:
pre_clasif = joblib.load(pp.RUTA_PREPROCESADORES / "clasificacion.joblib")
num = pre_clasif.named_steps["columnas"].named_transformers_["num"]
bin_ = pre_clasif.named_steps["columnas"].named_transformers_["bin"]
cols_num = pre_clasif.named_steps["columnas"].transformers_[0][2]
cols_bin = pre_clasif.named_steps["columnas"].transformers_[1][2]
aprendido = pd.DataFrame({
    "mediana imputación": num.named_steps["simpleimputer"].statistics_,
    "media escalado": num.named_steps["standardscaler"].mean_,
    "desv. escalado": num.named_steps["standardscaler"].scale_,
}, index=cols_num)
display(aprendido)
pd.Series(bin_.statistics_, index=cols_bin, name="moda imputación").to_frame().T

,mediana imputación,media escalado,desv. escalado
imc,28.0,29.065882,6.861956
dias_salud_mental_mala,0.0,3.372549,7.811519
dias_salud_fisica_mala,0.0,4.990098,9.435280
grupo_edad,9.0,8.390000,2.964572
salud_general,3.0,2.694412,1.100823
nivel_educativo,5.0,4.968137,1.005994
nivel_ingresos,6.0,5.864216,2.077823
imc_categoria,3.0,3.355882,1.161986
n_factores_riesgo_cv,2.0,1.614706,1.203744


,presion_alta,colesterol_alto,chequeo_colesterol_5a,fumador,ictus,enfermedad_cardiaca,actividad_fisica,consume_fruta,consume_verdura,consumo_alcohol_alto,dificultad_caminar,es_hombre,falta_habitos,falta_imc
moda imputación,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0


In [10]:
X_train_c = pd.read_parquet(pp.RUTA_PROCESADOS / "clasificacion" / "X_train.parquet")
print("Las numéricas quedan centradas y con varianza unitaria en train:")
display(X_train_c[cols_num].agg(["mean", "std"]).round(3))
X_train_c.head()

Las numéricas quedan centradas y con varianza unitaria en train:


,imc,dias_salud_mental_mala,dias_salud_fisica_mala,grupo_edad,salud_general,nivel_educativo,nivel_ingresos,imc_categoria,n_factores_riesgo_cv
mean,-0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


,imc,dias_salud_mental_mala,dias_salud_fisica_mala,grupo_edad,salud_general,nivel_educativo,nivel_ingresos,imc_categoria,n_factores_riesgo_cv,presion_alta,colesterol_alto,chequeo_colesterol_5a,fumador,ictus,enfermedad_cardiaca,actividad_fisica,consume_fruta,consume_verdura,consumo_alcohol_alto,dificultad_caminar,es_hombre,falta_habitos,falta_imc
id_paciente_proy,,,,,,,,,,,,,,,,,,,,,,,
10186,-0.446794,-0.43174,-0.528877,1.217714,0.277600,-1.956410,1.027895,-0.306271,1.150821,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0
3593,-1.175449,-0.43174,-0.528877,-0.131554,-1.539222,1.025715,-0.415924,-1.166867,-1.341403,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
10944,-0.301063,-0.43174,-0.528877,0.543080,-0.630811,0.031673,0.065349,-0.306271,-0.510662,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
13695,-1.904105,-0.43174,0.001049,-0.468870,1.186011,1.025715,0.546622,-2.027463,-0.510662,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
881,-0.446794,-0.43174,-0.316906,0.543080,0.277600,0.031673,1.027895,-0.306271,0.320080,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0


## 5. Clustering

No hay variable objetivo ni evaluación sobre datos retenidos, así que el preprocesador se ajusta con **todos** los registros. Se usan las 12 variables de salud y hábitos (`pp.COLS_CLUSTERING`); las binarias también se estandarizan para que todas pesen igual en las distancias. Las sociodemográficas quedan fuera para **describir** los clústeres después (a partir de `diabetes_limpio`).

In [11]:
pre_clu = pp.construir_preprocesador("clustering").fit(crudo)
X_clu = pre_clu.transform(crudo)
X_clu.to_parquet(pp.RUTA_PROCESADOS / "clustering" / "X_clustering.parquet")
joblib.dump(pre_clu, pp.RUTA_PREPROCESADORES / "clustering.joblib")
print(f"X_clustering: {X_clu.shape} · nulos: {X_clu.isna().sum().sum()}")
X_clu.agg(["mean", "std"]).round(3)

X_clustering: (12750, 12) · nulos: 0


,imc,dias_salud_fisica_mala,dias_salud_mental_mala,grupo_edad,salud_general,presion_alta,colesterol_alto,diabetes,enfermedad_cardiaca,dificultad_caminar,actividad_fisica,fumador
mean,0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,0.0,0.0,0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 6. Prueba de reproducibilidad (de cara a la API)

El profesor evaluará los modelos enviando **instancias crudas** a la API. Comprobamos que el preprocesador guardado, aplicado a filas crudas del CSV, reproduce exactamente la matriz guardada, y que tolera codificaciones nuevas o valores ausentes sin romperse.

In [12]:
pre = joblib.load(pp.RUTA_PREPROCESADORES / "clasificacion.joblib")
X_test_guardado = pd.read_parquet(pp.RUTA_PROCESADOS / "clasificacion" / "X_test.parquet")

muestra = crudo[crudo[pp.COL_ID].isin(X_test_guardado.index[:200])].drop(columns=pp.TARGET_CLASIF)
recalculado = pre.transform(muestra)
pd.testing.assert_frame_equal(recalculado, X_test_guardado.loc[recalculado.index], check_names=False)
print("✔ El preprocesador guardado reproduce exactamente X_test a partir de los datos crudos.")

# Instancia "difícil": codificaciones no vistas, nulos y sin columnas PII
rara = muestra.iloc[[0]].drop(columns=pp.COLS_PII).copy()
rara[["fumador", "actividad_fisica", "salud_general", "imc"]] = ["Sí", "yes", " Muy Buena ", np.nan]
print("✔ Instancia con codificaciones no vistas y nulos transformada sin errores:")
pre.transform(rara)

✔ El preprocesador guardado reproduce exactamente X_test a partir de los datos crudos.
✔ Instancia con codificaciones no vistas y nulos transformada sin errores:


,imc,dias_salud_mental_mala,dias_salud_fisica_mala,grupo_edad,salud_general,nivel_educativo,nivel_ingresos,imc_categoria,n_factores_riesgo_cv,presion_alta,colesterol_alto,chequeo_colesterol_5a,fumador,ictus,enfermedad_cardiaca,actividad_fisica,consume_fruta,consume_verdura,consumo_alcohol_alto,dificultad_caminar,es_hombre,falta_habitos,falta_imc
id_paciente_proy,,,,,,,,,,,,,,,,,,,,,,,
2234,-0.155332,-0.43174,-0.528877,0.54308,-0.630811,1.025715,-0.415924,-0.306271,1.150821,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


## 7. Resumen de artefactos generados

In [13]:
artefactos = sorted(p for p in (pp.RUTA_PROCESADOS.rglob("*") ) if p.is_file()) + sorted(pp.RUTA_PREPROCESADORES.glob("*.joblib"))
filas = []
for p in artefactos:
    forma = pd.read_parquet(p).shape if p.suffix == ".parquet" else ""
    filas.append({"archivo": str(p.relative_to(pp.RUTA_PROYECTO)).replace("\\", "/"), "dimensiones": forma, "tamaño (KB)": round(p.stat().st_size / 1024, 1)})
pd.DataFrame(filas).set_index("archivo")

,dimensiones,tamaño (KB)
archivo,,
data/processed/clasificacion/X_test.parquet,"(2550, 23)",45.5
data/processed/clasificacion/X_train.parquet,"(10200, 23)",134.5
data/processed/clasificacion/y_test.parquet,"(2550, 1)",16.3
data/processed/clasificacion/y_train.parquet,"(10200, 1)",60.9
data/processed/clustering/X_clustering.parquet,"(12750, 12)",129.1
data/processed/diabetes_limpio.csv,,1259.0
data/processed/diabetes_limpio.parquet,"(12750, 24)",172.0
data/processed/regresion/X_test.parquet,"(2550, 23)",44.0
data/processed/regresion/X_train.parquet,"(10200, 23)",128.4


**Estado para el punto de control 1**
- ✔ Datos limpios y documentados, con trazabilidad por `id_paciente_proy`.
- ✔ Conjuntos de entrenamiento y test listos para **clasificación** y **regresión**, sin fugas (preprocesador ajustado solo en train).
- ✔ Matriz estandarizada lista para **clustering**.
- ✔ Preprocesadores serializados que parten de datos crudos → reutilizables directamente en la API.

**Pendiente para las siguientes fases**: selección de variables por línea (p. ej. descartar `consume_fruta`/`consume_verdura` en regresión, usar derivadas *o* componentes en modelos lineales), modelado, validación cruzada e hiperparametrización. Cuando se publique el contrato de la API se confirmará qué columnas llegan en cada petición (p. ej. si `dias_salud_fisica_mala` se envía en las instancias de clasificación) y se ajustará `pp.columnas_features` si hace falta.